## Price Prediction Pipeline

In [3]:
import pandas as pd


market_df = pd.read_csv("HINDUNILRVR_market_data.csv")
sentiment_df = pd.read_csv("HINDUNILIVR_sentiment_data.csv")

Here I am preparing the dataset by combining the tweet data and the respective OHLCV data and the corresponding next day return

In [4]:
merged_df = pd.merge(market_df, sentiment_df, on="timestamp", how="inner")

In [5]:
merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)

In [6]:
print(merged_df.head())
print(f"\nMerged dataset shape: {merged_df.shape}")

    timestamp        open        high         low       close   adj_close  \
0  2025-01-01  100.496714  100.831679  100.058577   99.890014   99.792676   
1  2025-01-02  100.358450  100.567586  100.285007  100.569734  100.670447   
2  2025-01-03  101.006138  102.278999  100.790087  102.206217  102.239832   
3  2025-01-04  102.529168  103.942120  101.851447  102.037266  102.014518   
4  2025-01-05  102.295015  102.358187  100.689844  100.418462  100.336069   

   volume  next_close  next_adj_close  management_sent  governance_sent  \
0    3173  100.569734      100.670447         0.611528        -0.150343   
1    4479  102.206217      102.239832         0.650497        -0.743630   
2    2588  102.037266      102.014518         0.736406        -0.754156   
3    2221  100.418462      100.336069        -0.171737         0.239136   
4    1810  102.680589      102.608678         0.395729         0.351065   

   fundamentals_sent  hype_sent      fear       joy     anger     trust  
0          -

In [7]:
merged_df.to_csv("HINDUNILIVR__merged_data.csv", index=False)

## Preparing the target for the prediction which is next day return

In [8]:
merged_df["next_day_return"] = (merged_df["next_adj_close"] - merged_df["adj_close"]) / merged_df["adj_close"]


merged_df = merged_df.dropna().reset_index(drop=True)


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

Training the model to learn patterns based on previous day market data,and corresponding sentiments and emotion features

In [ ]:
target = "next_day_return"
X = merged_df.drop(columns=["timestamp", target])
y = merged_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Test MSE: {mse:.6f}")


results = X_test.copy()
results["actual_return"] = y_test.values
results["predicted_return"] = y_pred

print(results.head())

Test MSE: 0.000163
          open       high        low      close  adj_close  volume  \
799  93.681767  95.267475  92.336846  92.295779  92.160027    3413   
800  94.620050  95.775237  92.666037  95.125640  95.035573    2313   
801  94.104006  95.373171  93.900220  95.593119  95.496699    2897   
802  94.200127  95.795955  94.183487  96.471576  96.579582    1402   
803  93.737851  94.529792  92.870686  93.333454  93.427733    3105   

     next_close  next_adj_close  management_sent  governance_sent  \
799   95.125640       95.035573         0.143255        -0.927585   
800   95.593119       95.496699        -0.907194         0.102283   
801   96.471576       96.579582         0.247919         0.417362   
802   93.333454       93.427733         0.252196        -0.771124   
803   93.794784       93.843515         0.782402        -0.389134   

     fundamentals_sent  hype_sent      fear       joy     anger     trust  \
799          -0.937277   0.735564  0.204455  0.774550  0.954666  0.2

In [15]:
results["residual"] = results["actual_return"] - results["predicted_return"]


window = 20
results["rolling_sigma"] = results["residual"].rolling(window=window, min_periods=1).std()


k = 2  
results["lower_bound"] = results["predicted_return"] - k * results["rolling_sigma"]
results["upper_bound"] = results["predicted_return"] + k * results["rolling_sigma"]

results["anomaly_flag"] = (results["actual_return"] < results["lower_bound"]) | \
                          (results["actual_return"] > results["upper_bound"])


print(results[["actual_return", "predicted_return", "residual", "rolling_sigma",
               "lower_bound", "upper_bound", "anomaly_flag"]].head(25))

     actual_return  predicted_return  residual  rolling_sigma  lower_bound  \
799       0.031202          0.022271  0.008930            NaN          NaN   
800       0.004852          0.006649 -0.001797       0.007585    -0.008521   
801       0.011339          0.010722  0.000617       0.005627    -0.000533   
802      -0.032635         -0.025146 -0.007489       0.006817    -0.038780   
803       0.004450          0.006796 -0.002346       0.006002    -0.005207   
804      -0.001837          0.002310 -0.004147       0.005580    -0.008849   
805      -0.003707         -0.000363 -0.003345       0.005168    -0.010698   
806      -0.006968         -0.007264  0.000296       0.004820    -0.016905   
807       0.018810          0.014291  0.004519       0.004890     0.004511   
808      -0.039883         -0.032307 -0.007576       0.005121    -0.042549   
809       0.032335          0.030179  0.002157       0.004965     0.020250   
810      -0.010927         -0.008297 -0.002630       0.004759   